Send Forecast Email
Reads the latest generated forecast and emails a readable summary to the ERP team for validation.

**Input**: gold/live/forecasts/overall_forecast_latest.json
**Output**: email sent

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service
import json
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

blob_service = get_blob_service(storage_account_name, storage_account_key)

In [0]:
# Overall — read from active (contains all not-yet-ended forecasts, deduped to latest per period)
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly = json.loads(stream)

body = """Hello,

Here is the latest battery sales forecast.

WEEKLY FORECAST
"""
for w in active_weekly:
    body += f"Week starting {w['week_start']}: {w['predicted_units']:,} units (generated {w['generated_date']})\n"

body += "\nMONTHLY FORECAST\n"
for m in active_monthly:
    body += f"{m['month_start']}: {m['predicted_units']:,} units (range: {m['lower_bound']:,} - {m['upper_bound']:,}, generated {m['generated_date']})\n"

# Brand — read from active
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/active/brand_weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_weekly = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/active/brand_monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_brand_monthly = json.loads(stream)

body += "\nBRAND BREAKDOWN (Weekly)\n"
for w in active_brand_weekly:
    body += f"{w['brand_code']} — Week starting {w['week_start']}: {w['predicted_units']:,} units\n"

body += "\nBRAND BREAKDOWN (Monthly)\n"
for m in active_brand_monthly:
    body += f"{m['brand_code']} — {m['month_start']}: {m['predicted_units']:,} units\n"

body += """
Please review and let us know if these numbers look reasonable based on your knowledge of current orders/promotions.

This is an automated message from the Exide Sales Forecasting pipeline.
"""

print(body)

In [0]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib

msg = MIMEMultipart()
msg["From"] = smtp_username
msg["To"] = ", ".join(to_emails)
msg["Subject"] = f"Battery Sales Forecast — {forecast['generated_at'][:10]}"
msg.attach(MIMEText(body, "plain"))

try:
    server = smtplib.SMTP(smtp_server, smtp_port)
    server.starttls()

    server.login(smtp_username, smtp_password)

    # Send to all recipients
    server.sendmail(
        smtp_username,
        to_emails,
        msg.as_string()
    )

    print(f"Email sent to {', '.join(to_emails)}")

except Exception as e:
    print(f"Failed to send email: {e}")

finally:
    server.quit()